In [1]:
import pandas as pd

df = pd.read_csv('data/train.csv')
print('Shape of train.csv:', df.shape)

Shape of train.csv: (108954, 3)


In [2]:
df.head

<bound method NDFrame.head of             id                                              input  \
0        45361  হেলো, আমরা গত ৩ মাস ধরে গর্ভধারণের চেষ্টা করছি...   
1        95550  প্রায় এক মাস আগে আমার পিঠের ডান দিকের কাঁধের হ...   
2        98599  হেলো, আমি ২৫ বছর বয়সী অবিবাহিত মেয়ে। আমার কপাল...   
3        84959  হেলো, আমার বয়স ২২, উচ্চতা ৫ ফুট ৯ ইঞ্চি এবং ওজ...   
4        65169  হেলো, আমার নানি প্রায়ই অভিযোগ করেন যে তার হৃদপ...   
...        ...                                                ...   
108949   78957  আমার ৮ বছর বয়সী ছেলে গত রাতে তার বগলে ব্যথার ক...   
108950  111475  আমার বয়স ৪০-এর বেশি এবং গত কয়েক বছর ধরে আমার চ...   
108951  104289  হেলো ডাক্তার, আমার মায়ের বয়স ৮২ বছর। গত পনেরো ...   
108952     861  স্যার, আক্কেল দাঁতে ক্যাভিটি থাকার কারণে কি টি...   
108953   15817  আমার পাঁচ বছর বয়সী মেয়ে তিন দিন ধরে পেটে ব্যথা...   

                                                   output  
0       হেলো, পিসিওডি (PCOD) হলো এমন একটি উপসর্গ যা অন...  
1       হেলো, প্রথমে 

## Convert to Parquet

CSV round-trips slowly and `input`/`output` contain embedded newlines/commas inside quotes — that's why a raw line count of the file (~119.8k) doesn't match the real row count (108,954): the gap is multi-line quoted text, not extra rows. Converting once to Parquet now gives a lossless, ~4x faster, ~63% smaller working copy for every later step. `train.csv` (and `test.csv`) are left untouched; all cleaning steps below read/write `train_raw.parquet` and its successors instead.

In [ ]:
import time

t0 = time.time()
df.to_parquet('data/train_raw.parquet', index=False, engine='pyarrow')
print(f'Wrote parquet in {time.time()-t0:.1f}s')

# Integrity check: round-trip must match the CSV-parsed frame exactly
df_check = pd.read_parquet('data/train_raw.parquet')
assert df.equals(df_check), 'Mismatch between CSV load and Parquet round-trip!'
print('Round-trip verified: identical data.')

print('\nNulls per column:\n', df.isnull().sum())
print('\nDuplicate ids:', df['id'].duplicated().sum())
print('Duplicate (input, output) pairs:', df.duplicated(subset=['input', 'output']).sum())

## Train / validation split (before any cleaning)

Held out **10%** (10,895 rows) as `val_split.parquet`, seed=42, disjoint ids from the remaining **90%** (`train_split.parquet`, 98,059 rows). Input/output length distributions match closely between the two, so the split is representative. Splitting now, before cleaning, means every later step runs identically on both halves, and nothing computed across the *whole* corpus (dedup, length cutoffs, vocab/frequency stats, etc.) can leak from val into train. From here on, preprocessing reads/writes `train_split.parquet`; `val_split.parquet` gets the same transformations applied but must never be used to *derive* a cleaning decision — only `train_split` drives those.

In [ ]:
VAL_FRAC = 0.10
SEED = 42

df_raw = pd.read_parquet('data/train_raw.parquet')

val = df_raw.sample(frac=VAL_FRAC, random_state=SEED)
train = df_raw.drop(val.index)

assert len(set(train['id']) & set(val['id'])) == 0
assert len(train) + len(val) == len(df_raw)

train = train.reset_index(drop=True)
val = val.reset_index(drop=True)
train.to_parquet('data/train_split.parquet', index=False)
val.to_parquet('data/val_split.parquet', index=False)

print(f'Train: {train.shape}, Val: {val.shape}')